# Resource-Aware Deep Learning: Hands-On

This notebook will teach you, how to track your own (deep-learning) pipeline's resource consumption using tools like [↗CodeCarbon](https://codecarbon.io/), [↗MLFlow](https://mlflow.org/), and the [↗Lamarr Energy Tracker (LET)](https://github.com/lamarr-institute/lamarr-energy-tracker). This tutorial is built on [↗PyTorch](https://pytorch.org) and [↗PyTorch Lightning](https://lightning.ai/docs/pytorch/stable/). Please make sure you have installed and activated the provided virtual environment (either through `(uv) venv` or `conda/mamba`).

We have provided a small Python package `resource_aware_ml` that implements a number of SRResNet architectures inside the `resource_aware_ml.architectures.model` submodule. If you have not yet installed the package, please do so now inside the repository root directory, e.g. using [uv](https://docs.astral.sh/uv/):

```shell-session
$ uv pip install -e .
```

We can now load the `models` submodule, which implements the following architectures in decreasing complexity:
```
SRResNet18
SRResNet10
SRResNet6
SRResNet4
```
![SRResNet Overview](assets/resnet_sc.png)

In [ ]:
from rich import print  # nicer prints

from resource_aware_ml.architectures import models

Choose one of the architectures. Beware that higher complexity will mean more GPU memory consumption. Make sure **not** to initialise the model. This will be handled by the `TrainModule` below.

In [ ]:
# Choose your architecture here
model = models.SRResNet10

print(model())

Before we go over the training loop, we have to import and initialize a training module. The training module sets up the training, validation, and inference steps of the model, as well as the optimizer. During the training and validation steps it also calls the loss function `loss_fn` that is used to monitor the training process.

In [ ]:
from torch.nn import L1Loss
from torch.optim import AdamW

from resource_aware_ml.training.trainer import TrainModule

In [ ]:
# Change the Optimizer or loss function here.
train_config = dict(model=model, loss_fn=L1Loss, optimizer=AdamW, lr=1e-3)

train_module = TrainModule(train_config)

We will also need a data module that handles the data loading for us. The data module implemented in the package is designed to load the dataset that is provided in the repository. The data module inherits from the [↗`lightning.LightningDataModule`](https://lightning.ai/docs/pytorch/stable/api/lightning.pytorch.core.LightningDataModule.html) and is simply a collection of dataloaders for the training, validation, test, and inference stages.

In [ ]:
from resource_aware_ml.io.data import H5DataModule

In [ ]:
data_module = H5DataModule(
    data_dir="./data",  # Path to the data in the repository relative to this notebook
    batch_size=20,  # Change if you run out of memory
    fourier=True,  # Our data is in Fourier space
    num_workers=4,  # Change the number of concurrent CPU cores loading the data
)

## Task 1: Logging

Implement a logger that logs the experiment data such as the train loss and validation loss to **MLFlow**. Have a look at the lightning docs for more information on [loggers](https://lightning.ai/docs/pytorch/stable/api_references.html#loggers). Save the logger in the list `logger`.

In [ ]:
from pathlib import Path

from lightning.pytorch.loggers import CSVLogger, MLFlowLogger


mlruns_dir = Path("./build/mlruns").expanduser().resolve()
logger = [
    CSVLogger(save_dir="./build", name="training_logs"),
    # ADD THE MLFlow LOGGER HERE
    # ==========================
]

Now we can set up the [↗`lightning.Trainer`](https://lightning.ai/docs/pytorch/stable/api/lightning.pytorch.trainer.trainer.Trainer.html). Here, we can set

In [ ]:
from lightning import Trainer
from lightning.pytorch.callbacks import RichProgressBar


trainer = Trainer(
    max_epochs=10,
    accelerator="auto",  # Change to gpu, or cpu, if you like
    precision="32-true",
    logger=logger,
    callbacks=RichProgressBar(),
    log_every_n_steps=20,  # set to batch size
)

## Task 2: Training Emission Tracking

To start the training process, we will have to use the `Trainer.fit` method. In order to track the emissions, we will need to use CodeCarbon or the Lamarr Energy Tracker (LET).

In [ ]:
import numpy as np
from lamarr_energy_tracker import EnergyTracker

from resource_aware_ml.utils import get_predictions

## 

Your task is to log the following metrics and parameters using MLFlow and CodeCarbon/LET:
<table>
<tr><td>
    
| Metric                         | Variable Name              |
| ------------------------------ | -------------------------- |
| Number of trainable parameters | `num_trainable_parameters` |
| Total runtime                  | `running_time_total`       |
| Runtime per epoch              | `running_time`             |
| Total power draw               | `power_draw_total`         |
| Power draw per epoch           | `power_draw`               |
| Mean Total Flux Ratio          | `mean_total_flux`          |
| Mean Peak Flux Ratio           | `mean_peak_flux`           |
| Epochs trained                 | `epochs`                   |

</td><td>

| Parameter                | Variable Name  |
| ------------------------ | -------------- |
| Model name               | `model`        |
| Dataset                  | `dataset`      |
| Architecture (GPU Model) | `architecture` |
| Task (`training`)        | `task`         |

</td></tr> </table>

To get the predictions use the function
```python
resource_aware_ml.utils.get_predictions(trainer, task=Literal["training" | "inference"]) -> tuple[np.ndarray, np.ndarray] | np.ndarray
```
from the `resource_aware_ml` package. For the `training` task, this function returns predictions and targets.
The mean total flux ratio and the mean peak flux ratios can then be calculated as

$$
    R = \texttt{mean}\left(\frac{\texttt{preds}}{\texttt{targets}}\right).
$$

Values close to `1.0` would indicate good predictions. In order to use the ratios as suitable metrics for STREP, minimize the ratios
as
```python
mean_<metric>_flux = abs(1.0 - <metric>)
```

***Hint***: Use the `.sum()` and `.max()` methods to get the total and peak fluxes from the data.

In [ ]:
emissions_path = Path("./build").expanduser().resolve()

# Track the training loop
# ADD YOUR CODE HERE =======




# ==========================

# When using multiple loggers, get the experiment and
# run id from the MLFLowLogger instance
mlflow_logger = next(
    logger for logger in trainer.loggers if isinstance(logger, MLFlowLogger)
)
experiment = mlflow_logger.experiment
run_id = mlflow_logger._run_id

# Get total number of samples (train + valid)
num_samples = trainer.datamodule.train_length + trainer.datamodule.valid_length

# Get predictions and targets
preds, targets = get_predictions(trainer, task="training")
# ADD YOUR CODE HERE =======





# ==========================


# get results from tracker
last_results = tracker.results.iloc[-1]

metrics = dict(
    # ADD YOUR CODE HERE =======
    
    
    
        
    # ==========================
)


for key, val in metrics.items():
    experiment.log_metric(
        key=key,
        value=val,
        run_id=run_id,
    )


model_name = # ADD YOUR CODE HERE

params = dict(
    # ADD YOUR CODE HERE =======
    
    
    
        
    # ==========================
)


for key, val in params.items():
    experiment.log_param(
        key=key,
        value=val,
        run_id=run_id,
    )

## Task 3: Inference Emission Tracking

Your task is to log the following metrics and parameters using MLFlow and CodeCarbon/LET for inference:
<table>
<tr><td>
    
| Metric                         | Variable Name              |
| ------------------------------ | -------------------------- |
| Number of trainable parameters | `num_trainable_parameters` |
| Total runtime                  | `running_time_total`       |
| Runtime per epoch              | `running_time`             |
| Total power draw               | `power_draw_total`         |
| Power draw per epoch           | `power_draw`               |
| Mean Total Flux Ratio          | `mean_total_flux`          |
| Mean Peak Flux Ratio           | `mean_peak_flux`           |
| Epochs trained                 | `epochs`                   |

</td><td>

| Parameter                | Variable Name  |
| ------------------------ | -------------- |
| Model name               | `model`        |
| Dataset                  | `dataset`      |
| Architecture (GPU Model) | `architecture` |
| Task (`training`)        | `task`         |


</td></tr> </table>

In [ ]:
# Change checkpoint path accordingly
train_module = TrainModule.load_from_checkpoint(
    "./build/training_logs/version_0/checkpoints/epoch=9-step=110.ckpt",
    weights_only=False,
)

# Reinitialize MLFlowLogger and Trainer, otherwise we will get an error
logger = [
    CSVLogger(save_dir="./build", name="inference_logs"),
    MLFlowLogger(save_dir=mlruns_dir),
]
trainer = Trainer(
    max_epochs=10,
    accelerator="auto",  # Change to gpu, or cpu, if you like
    precision="32-true",
    logger=logger,
    callbacks=RichProgressBar(),
    log_every_n_steps=20,  # set to batch size
)

emissions_path = Path("./build").expanduser().resolve()

# Track the prediction
# ADD YOUR CODE HERE =======




# ==========================

# When using multiple loggers, get the experiment and
# run id from the MLFLowLogger instance
mlflow_logger = next(
    logger for logger in trainer.loggers if isinstance(logger, MLFlowLogger)
)
experiment = mlflow_logger.experiment
run_id = mlflow_logger._run_id

num_samples = trainer.datamodule.predict_length

# Get predictions and targets
# NOTE: We're using the 'testing' mode here since we have a
# testing dataset with ground truths. In true inference
# We could not compute the metrics below.
preds, targets = get_predictions(trainer, task="testing")
# ADD YOUR CODE HERE =======





# ==========================

# Get results from tracker
last_results = tracker.results.iloc[-1]

metrics = dict(
    # ADD YOUR CODE HERE =======
    
    
    
        
    # ==========================
    train_loss=1,  # Since we are writing to the same file as
    val_loss=1,  # the training run, we have to log both losses.
    epoch=trainer.current_epoch,  # In this case, we also have to log the epochs
)

for key, val in metrics.items():
    experiment.log_metric(
        key=key,
        value=val,
        run_id=run_id,
    )

model_name = # ADD YOUR CODE HERE

params = dict(
    # ADD YOUR CODE HERE =======
    
    
    
        
    # ==========================
)

for key, val in params.items():
    experiment.log_param(
        key=key,
        value=val,
        run_id=run_id,
    )

## Task 4: Train More Models

Repeat tasks 2 and 3 for different model configurations, e.g. by changing the architecture, the optimizer, or the loss function.

## Task 5: MLFlow Export

Export the logged data to a `.csv` file using MLFlow's CLI tool.

## Task 6: Visualisation Using STREP

Visualise the data using the STREP framework.